# Assignment 2B — Retrieval-Augmented Generation (RAG) Pipeline
**Group No. 7**  
**Course:** LLM4GenAI  
**Domain:** Financial Annual Reports (Apple, Amazon, NVIDIA, Tesla, Berkshire Hathaway)

---

## Pipeline Overview
```
Domain .txt Corpus
       ↓
  Part A: Chunking (Fixed-Size / Sliding Window / Semantic)
       ↓
  Part B: Retrieval (Dense FAISS / Sparse BM25 / Hybrid RRF)
       ↓
  Part C1: Cross-Encoder Reranking
  Part C2: Tabular RAG (PDF tables → serialised rows → indexed)
```

---
## 📦 Step 1.1 — Install Dependencies

Install all required libraries. Run this cell first.
- `sentence-transformers` — for dense embeddings
- `faiss-cpu` — vector index for dense retrieval
- `rank_bm25` — BM25 sparse retrieval
- `pdfplumber` — extract tables from PDFs
- `transformers` — cross-encoder reranking model
- `nltk` — sentence tokenisation for semantic chunking

In [ ]:
import sys

# Install all required packages
!{sys.executable} -m pip install sentence-transformers faiss-cpu rank_bm25 pdfplumber \
    pandas numpy transformers nltk tqdm --quiet

print("✅ All dependencies installed successfully.")

---
## 📂 Step 1.2 — Load Corpus

Unzip the domain corpus from Assignment 1A. It contains 5 cleaned financial annual report
text files: Apple, Amazon, NVIDIA, Tesla, and Berkshire Hathaway.
We load each file separately so we can track which company each chunk comes from.

In [ ]:
import zipfile
import os
import pandas as pd

# Extract corpus zip into a local folder
CORPUS_ZIP = "ASSIGNMENT1stuff/domain_corpus (2).zip"
CORPUS_DIR = "corpus"

os.makedirs(CORPUS_DIR, exist_ok=True)

with zipfile.ZipFile(CORPUS_ZIP, 'r') as z:
    z.extractall(CORPUS_DIR)

# Load each .txt file and record word count
corpus_files = {}
stats = []

for fname in sorted(os.listdir(CORPUS_DIR)):
    if fname.endswith('.txt'):
        fpath = os.path.join(CORPUS_DIR, fname)
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        company = fname.replace('.txt', '')
        word_count = len(text.split())
        corpus_files[company] = text
        stats.append({'Company': company, 'File': fname, 'Words': word_count, 'Chars': len(text)})
        print(f"  Loaded {company:<15} → {word_count:>7,} words")

# Combine all documents into one corpus string
full_corpus = "\n\n".join(corpus_files.values())
total_words = sum(s['Words'] for s in stats)

print(f"\n{'='*45}")
print(f"  Total documents : {len(corpus_files)}")
print(f"  Total words     : {total_words:,}")
print(f"  Total chars     : {len(full_corpus):,}")
print(f"{'='*45}")

df_corpus_stats = pd.DataFrame(stats)
display(df_corpus_stats)

---
## 🌐 Step 1.3 — Download Annual Report PDFs (for Tabular RAG in Part C)

We re-download the original 5 financial annual report PDFs from Assignment 1B.
These are needed for Part C2 (Tabular RAG) where we extract tables using pdfplumber.
Financial PDFs are rich in structured tables: income statements, balance sheets, segment data.

**Note:** If any download fails (network/size issues), we fall back to using alternate
publicly available financial PDFs — the assignment explicitly permits this.

In [ ]:
import urllib.request
import time

PDF_DIR = "domain_pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# Original PDF URLs from Assignment 1B
PDF_URLS = {
    'apple':      'https://d18rn0p25nwr6d.cloudfront.net/CIK-0000320193/b4266e40-1de6-4a34-9dfb-8632b8bd57e0.pdf',
    'amazon':     'https://s2.q4cdn.com/299287126/files/doc_financials/2024/ar/Amazon-com-Inc-2023-Annual-Report.pdf',
    'nvidia':     'https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf',
    'tesla':      'https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q4-2023-Update.pdf',
    'berkshire':  'https://www.berkshirehathaway.com/2023ar/2023ar.pdf',
}

# SEC EDGAR requires a User-Agent header to avoid 403 errors
HEADERS = {
    'User-Agent': 'Assignment2B/1.0 2024ad05187@wilp.bits-pilani.ac.in',
    'Accept': 'application/pdf,*/*'
}

downloaded_pdfs = {}
failed_pdfs = []

for company, url in PDF_URLS.items():
    out_path = os.path.join(PDF_DIR, f"{company}.pdf")
    
    # Skip if already downloaded
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        size_mb = os.path.getsize(out_path) / 1e6
        print(f"  ✅ {company:<12} already exists ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path
        continue
    
    try:
        print(f"  ⬇️  Downloading {company}...", end=' ', flush=True)
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()
        
        # Verify it's actually a PDF
        if not data.startswith(b'%PDF'):
            raise ValueError("Response is not a valid PDF")
        
        with open(out_path, 'wb') as f:
            f.write(data)
        
        size_mb = len(data) / 1e6
        print(f"✅ ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path
        time.sleep(1)  # be polite to servers
        
    except Exception as e:
        print(f"❌ FAILED: {e}")
        failed_pdfs.append(company)

print(f"\n  Downloaded: {len(downloaded_pdfs)}/5 PDFs")
if failed_pdfs:
    print(f"  Failed: {failed_pdfs} — will use fallback PDFs for tabular extraction")

---
## ✅ Step 1.4 — Corpus Summary

Confirm corpus is loaded and ready. Log total word count and per-document breakdown.
This corpus will be used for all chunking and retrieval experiments in Parts A and B.

In [ ]:
# Final corpus summary before chunking
print("CORPUS READY FOR CHUNKING")
print("=" * 45)
for s in stats:
    bar = '█' * (s['Words'] // 5000)
    print(f"  {s['Company']:<12} {s['Words']:>8,} words  {bar}")
print("-" * 45)
print(f"  {'TOTAL':<12} {total_words:>8,} words")
print(f"\n  Estimated chunks @ 200 words (fixed-size): ~{total_words // 200:,}")
print(f"  Estimated chunks @ sliding window (+10%):  ~{int(total_words / 180):,}")
print("=" * 45)

---
## 📐 Part A — Chunking Strategies

We implement three chunking strategies on the 5-company financial corpus.
All chunkers operate **per company** using the `corpus_files` dict built in Step 1.2,
then merge results into a unified list with company metadata.

**Why per-company?** Chunking the concatenated string would create chunks that straddle
company boundaries, mixing e.g. Apple revenue data with Amazon revenue data in a single
chunk — poisoning both retrieval and the AI model's context.

Three strategies:
1. **Fixed-size** — word-count splits, no overlap
2. **Sliding window** — 200-word window, 20-word overlap
3. **Semantic** — sentence-boundary-aware (NLTK), max 200 words per chunk


In [ ]:
import nltk
import numpy as np

# Download NLTK sentence tokenizer data (punkt_tab for newer NLTK, punkt as fallback)
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass
nltk.download('punkt', quiet=True)

# ── Chunker 1: Fixed-size ─────────────────────────────────────────────────────
def fixed_size_chunker(text, max_words=200):
    """Split text into non-overlapping word-count windows."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

# ── Chunker 2: Sliding window ─────────────────────────────────────────────────
def sliding_window_chunker(text, max_words=200, overlap=20):
    """Split text with overlapping windows (step = max_words - overlap)."""
    words = text.split()
    step = max_words - overlap  # 180
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + max_words])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

# ── Chunker 3: Semantic (sentence-boundary-aware) ─────────────────────────────
def semantic_chunker(text, max_words=200):
    """
    Fill chunks greedily at sentence boundaries up to max_words.
    If a single sentence exceeds max_words, force-split at word boundary.
    """
    try:
        sentences = nltk.sent_tokenize(text)
    except LookupError:
        # Fallback: split on period+space if NLTK data unavailable
        import re
        sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_words = []
    current_len = 0

    for sent in sentences:
        sent_words = sent.split()
        sent_len = len(sent_words)

        if sent_len == 0:
            continue

        if sent_len > max_words:
            # Oversized sentence: flush current buffer first
            if current_words:
                chunks.append(" ".join(current_words))
                current_words, current_len = [], 0
            # Force-split the oversized sentence at word boundaries
            for i in range(0, sent_len, max_words):
                piece = " ".join(sent_words[i:i + max_words])
                if piece.strip():
                    chunks.append(piece)
            continue

        if current_len + sent_len > max_words:
            # Flush current buffer
            if current_words:
                chunks.append(" ".join(current_words))
            current_words = sent_words
            current_len = sent_len
        else:
            current_words.extend(sent_words)
            current_len += sent_len

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks

print("✅ Three chunkers defined: fixed_size_chunker, sliding_window_chunker, semantic_chunker")


---
## 📊 Step A1 — Apply Chunkers to Corpus

We run each strategy over the 5 companies separately, then merge into a flat list.
Each chunk entry carries `company` and `text` fields for downstream retrieval tagging.


In [ ]:
# Apply all three strategies per-company and merge
chunks_fixed    = []
chunks_sliding  = []
chunks_semantic = []

for company, text in corpus_files.items():
    for chunk in fixed_size_chunker(text):
        chunks_fixed.append({'company': company, 'text': chunk})
    for chunk in sliding_window_chunker(text):
        chunks_sliding.append({'company': company, 'text': chunk})
    for chunk in semantic_chunker(text):
        chunks_semantic.append({'company': company, 'text': chunk})

print(f"Fixed-size    : {len(chunks_fixed):,} chunks")
print(f"Sliding window: {len(chunks_sliding):,} chunks")
print(f"Semantic      : {len(chunks_semantic):,} chunks")


---
## 📊 Step A2 — Quality Metrics

Four metrics per strategy:
- **Total chunks** — how many pieces the corpus was split into
- **Avg size (words)** — mean chunk length
- **Std dev (words)** — how much chunk sizes vary
- **Broken sentences %** — chunks that do NOT end with `.`, `!`, `?`, `"`, or `)` 
  (i.e., cut mid-sentence)

Note: Tesla's metrics are less reliable due to its small size (~5,554 words → ~27 chunks)
and two-column PDF extraction artifacts that fragment its sentences regardless of strategy.


In [ ]:
def quality_metrics(chunk_list):
    """Compute quality metrics for a list of chunk dicts."""
    texts = [c['text'] for c in chunk_list]
    sizes = [len(t.split()) for t in texts]

    SENTENCE_ENDINGS = ('.', '!', '?', '"', ')', '."', '!"', '?"')
    broken = sum(1 for t in texts if not t.rstrip().endswith(SENTENCE_ENDINGS))

    return {
        'Total Chunks': len(texts),
        'Avg Size (words)': round(np.mean(sizes), 1),
        'Std Dev (words)': round(np.std(sizes), 1),
        'Broken Sent %': round(100 * broken / len(texts), 1)
    }

metrics = {
    'Fixed-Size'    : quality_metrics(chunks_fixed),
    'Sliding Window': quality_metrics(chunks_sliding),
    'Semantic'      : quality_metrics(chunks_semantic),
}

# Display as table
df_metrics = pd.DataFrame(metrics).T
df_metrics.index.name = 'Strategy'
print("\n=== Step A2 — Chunking Quality Metrics ===\n")
print(df_metrics.to_string())
display(df_metrics)


---
## ✅ Step A3 — Best Strategy Selection

**Selected strategy: Semantic Chunking**

Semantic chunking was chosen for Part B retrieval for the following reasons:

1. **Lowest broken sentence rate** (~5% vs ~65% for fixed-size and sliding window).
   Chunks end at natural sentence boundaries, preserving complete thoughts.

2. **Better embedding quality.** The `all-MiniLM-L6-v2` sentence-transformer encodes
   *meaning*. A coherent, complete sentence produces a more accurate vector than a
   fragment cut mid-thought. Higher embedding accuracy → more relevant retrieval.

3. **No information loss at boundaries.** Sliding window reduces loss through overlap,
   but still cuts mid-sentence. Semantic chunking avoids the problem entirely.

**Trade-off acknowledged:** Semantic chunks have higher std dev in size (sentences vary
in length), producing less uniform chunks than fixed-size. This is acceptable — retrieval
quality depends on semantic coherence, not chunk uniformity.

**Fixed-size** was rejected due to its ~65% broken sentence rate.  
**Sliding window** improves coverage via overlap but does not fix boundary coherence.

> All subsequent Parts (B, C1, C2) will use `chunks_semantic` as the text corpus.
